# V1 · OSSE 4 km: análisis de las 18 UTC y pronósticos a 60 min

Un ciclo de la OSSE con el LETKF-T de `WRF_TEnKF`. El prior son las 59 guesses de las 18 UTC
de la corrida de asimilación de 4 km (radares reales, ciclos de 5 min desde las 12 UTC); la
verdad es el miembro 60. Las observaciones son reflectividad sintética de la verdad en los
puntos de retícula con eco en algún miembro, hasta 16 km, raleadas cada 2 puntos y cada 2
niveles. Se comparan $N_t = 1, 2, 4$ (α_s = 1, R = 5 dBZ², L = 2 km) contra el prior a las
18 UTC y, en los pronósticos de 60 min, contra el pronóstico sin asimilación.

Las métricas y los dominios son los del capítulo 4: SS = 1 − M/M_ref con M agrupado sobre el
dominio antes del cociente, la reflectividad promediada en dBZ miembro a miembro, y los
dominios **alcanzado**, **tormenta** (columnas con máximo de la verdad ≥ 20 dBZ) y
**observado**. Los números vienen de `verif/reduce_osse.py`; este notebook solo los lee.

In [ ]:
import os, sys
sys.path.insert(0, f"/data/ra000007/{os.environ.get('USER', '')}/pylib")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import vcommon as vc

# REBUILD = True corre reduce_osse.py para los tiempos que falten (lee la salida completa de
# la OSSE: unos minutos por tiempo en un nodo prepost). False solo lee lo ya reducido.
REBUILD = False
if REBUILD:
    import subprocess
    subprocess.run([sys.executable, "reduce_osse.py", "--nproc", "32"], check=True)

df = vc.load_stats()
ss = vc.skill(df)
mk = vc.load_npz("masks.npz")
EXPS = vc.experiments(df)
LEADS = sorted(df.lead.unique())
print("verif:", vc.VERIF)
print("experimentos:", EXPS, "| plazos [min]:", LEADS)

## 1 · Diseño

In [ ]:
a0 = df[(df.time == "1800") & (df.exp == "prior") & (df["var"] == "ref")].set_index("domain")
k0, k1, j0, j1, i0, i1 = (int(b) for b in mk["box"])
design = pd.DataFrame({
    "celdas": a0.n.astype(int),
    "fracción del dominio": (a0.n / a0.loc["global", "n"]).round(3),
}).loc[list(vc.DOMAINS)]
print(f"observaciones sintéticas: {int(mk['n_obs'])}; caja de verificación: niveles {k0+1}-{k1} "
      f"(Fortran), {j1-j0} x {i1-i0} puntos; nivel de los mapas: {mk['zmean'][int(mk['lev5'])]/1e3:.2f} km")
design

## 2 · Análisis de las 18 UTC

Reflectividad: skill score contra el prior y fracción de las celdas que cambian que mejora.

In [ ]:
a = ss[(ss.time == "1800") & (ss["var"] == "ref")]
tab = a.pivot_table(index="exp", columns="domain", values=["ss_rmse", "ss_crps", "frac_imp"])
tab = tab.reindex(columns=list(vc.DOMAINS), level=1).loc[EXPS].round(3)
tab.index = [vc.exp_label(e, "es") for e in EXPS]
rmse_prior = df[(df.time == "1800") & (df.exp == "prior") & (df["var"] == "ref")].set_index("domain").rmse
print("RMSE del prior [dBZ]:", ", ".join(f"{d} {rmse_prior[d]:.2f}" for d in vc.DOMAINS))
tab

In [ ]:
def draw(lang):
    V = vc.VOCAB[lang]
    fig, axs = plt.subplots(1, 2, figsize=(vc.PAGE_W, 0.38 * vc.PAGE_W))
    a = ss[(ss.time == "1800") & (ss["var"] == "ref")]
    marks = {"global": "o", "reached": "s", "storm": "^", "observed": "D"}
    for ax, m, letter in zip(axs, ["ss_rmse", "ss_crps"], "ab"):
        for d in vc.DOMAINS:
            s = a[a.domain == d].assign(nt=lambda x: x.exp.map(vc.nt_of)).sort_values("nt")
            ax.plot(s.nt, s[m], marker=marks[d],
                    label=V[f"dom_{d}"])
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xticks(sorted(a.exp.map(vc.nt_of).unique()))
        ax.set_xlabel(V["nt"]); ax.set_ylabel(V[m])
        vc.panel_label(ax, letter)
    axs[0].legend(loc="best")
    fig.tight_layout()
    return fig

vc.save(draw, "V1_fig1_analysis_ss")

Variables de estado: $\mathrm{SS}_{\mathrm{RMSE}}$ contra el prior, en las celdas alcanzadas y en la tormenta.

In [ ]:
s = ss[ss.time == "1800"]
tabs = {}
for d in ("reached", "storm"):
    t = s[s.domain == d].pivot(index="var", columns="exp", values="ss_rmse")[EXPS]
    t.columns = [vc.exp_label(e, "es") for e in EXPS]
    tabs[d] = t.reindex(["ref", "qr", "qs", "qg", "w", "T", "u", "v", "P"]).round(3)
pd.concat(tabs, axis=1)

## 3 · Estructura espacial a las 18 UTC

In [ ]:
sl = vc.load_npz("slices_1800.npz")
lat, lon = mk["lat"], mk["lon"]
lev5 = int(mk["lev5"]); z5 = mk["zmean"][lev5] / 1e3

def draw(lang):
    V = vc.VOCAB[lang]
    cols = ["truth", "prior"] + EXPS
    fig, axs = plt.subplots(2, len(cols), figsize=(vc.PAGE_W, 0.5 * vc.PAGE_W), sharex=True, sharey=True)
    for c, name in enumerate(cols):
        ax = axs[0, c]
        pm = ax.pcolormesh(lon, lat, sl[f"{name}_ref5"], cmap=vc.DBZ_CMAP, norm=vc.DBZ_NORM,
                           shading="auto", rasterized=True)
        ax.set_title(V["truth"] if name == "truth" else vc.exp_label(name, lang))
        ax = axs[1, c]
        if name in EXPS:
            sk = ax.pcolormesh(lon, lat, sl[f"{name}_skill5"], cmap=vc.CMAP_DIV, vmin=-15, vmax=15,
                               shading="auto", rasterized=True)
        else:
            ax.axis("off")
    fig.colorbar(pm, ax=axs[0, :], shrink=0.8, label=V["ref"])
    fig.colorbar(sk, ax=axs[1, :], shrink=0.8, label=V["skill"])
    for ax in axs[:, 0]:
        ax.set_ylabel(V["lat"])
    for ax in axs[1, :]:
        ax.set_xlabel(V["lon"])
    for ax in axs.flat:
        ax.xaxis.set_major_locator(plt.MaxNLocator(3))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    fig.suptitle(f"{vc.fmt(z5, lang, '.1f')} km", y=1.0)
    return fig

vc.save(draw, "V1_fig2_maps_5km")

In [ ]:
jsec = int(mk["jsec"])
xlon = lon[jsec, :]
zkm = mk["zmean"] / 1e3

def draw(lang):
    V = vc.VOCAB[lang]
    cols = ["truth", "prior"] + EXPS
    fig, axs = plt.subplots(len(cols), 1, figsize=(vc.PAGE_W * 0.7, 0.22 * vc.PAGE_W * len(cols)),
                            sharex=True)
    for ax, name in zip(axs, cols):
        pm = ax.pcolormesh(xlon, zkm, sl[f"{name}_sec"], cmap=vc.DBZ_CMAP, norm=vc.DBZ_NORM,
                           shading="auto", rasterized=True)
        ax.set_ylabel(V["height"])
        ax.text(0.01, 0.95, V["truth"] if name == "truth" else vc.exp_label(name, lang),
                transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(fc="white", ec="none", alpha=0.7))
    axs[-1].set_xlabel(V["lon"])
    fig.colorbar(pm, ax=axs, shrink=0.6, label=V["ref"])
    fig.suptitle(f"{V['lat']} {vc.fmt(float(np.nanmean(lat[jsec])), lang, '.2f')}°", y=1.0)
    return fig

vc.save(draw, "V1_fig3_section")

## 4 · Pronósticos

Skill score de cada experimento contra el pronóstico sin asimilación del mismo plazo (a los
0 min, contra el prior). Las celdas alcanzadas y observadas son las de las 18 UTC; la tormenta
se recalcula con la verdad de cada plazo.

In [ ]:
def draw(lang):
    V = vc.VOCAB[lang]
    s = ss[ss["var"] == "ref"]
    fig, axs = plt.subplots(2, 4, figsize=(vc.PAGE_W, 0.5 * vc.PAGE_W), sharex=True, sharey="row")
    for r, m in enumerate(["ss_rmse", "ss_crps"]):
        for c, d in enumerate(vc.DOMAINS):
            ax = axs[r, c]
            for e in EXPS:
                x = s[(s.exp == e) & (s.domain == d)].sort_values("lead")
                ax.plot(x.lead, x[m], color=vc.exp_color(e), marker=".", label=vc.exp_label(e, lang))
            ax.axhline(0, color="k", lw=0.6)
            if r == 0:
                ax.set_title(V[f"dom_{d}"])
            if c == 0:
                ax.set_ylabel(V[m])
            if r == 1:
                ax.set_xlabel(V["lead"])
    axs[0, 0].legend(loc="best")
    fig.tight_layout()
    return fig

vc.save(draw, "V1_fig4_forecast_ss")

In [ ]:
def draw(lang):
    V = vc.VOCAB[lang]
    d0 = df[df["var"] == "ref"].copy()
    d0.loc[d0.exp == "prior", "exp"] = "noda"          # 0 min of the reference curve
    fig, axs = plt.subplots(2, 2, figsize=(vc.PAGE_W, 0.6 * vc.PAGE_W), sharex=True)
    for c, d in enumerate(["global", "storm"]):
        for e in ["noda"] + EXPS:
            x = d0[(d0.exp == e) & (d0.domain == d)].sort_values("lead")
            kw = dict(color=vc.exp_color(e), marker=".", label=vc.exp_label(e, lang))
            axs[0, c].plot(x.lead, x.rmse, **kw)
            axs[1, c].plot(x.lead, x.spread / x.rmse, **kw)
        axs[0, c].set_title(V[f"dom_{d}"])
        axs[1, c].set_xlabel(V["lead"])
        axs[1, c].axhline(1, color="k", lw=0.6)
    axs[0, 0].set_ylabel(V["rmse"] + " [dBZ]")
    axs[1, 0].set_ylabel(V["spread_ratio"])
    axs[0, 0].legend(loc="best")
    fig.tight_layout()
    return fig

vc.save(draw, "V1_fig5_forecast_rmse_spread")

Variables de estado en la tormenta: $\mathrm{SS}_{\mathrm{RMSE}}$ contra el pronóstico sin asimilación.

In [ ]:
pick = [l for l in (0, 15, 30, 60) if l in LEADS]
t = ss[(ss.domain == "storm") & ss.lead.isin(pick)].pivot_table(index="var", columns=["exp", "lead"], values="ss_rmse")
t = t.reindex(columns=EXPS, level=0).reindex(["ref", "qr", "qs", "qg", "w", "T", "u", "v", "P"]).round(3)
t

## 5 · Perfil vertical a las 18 UTC

In [ ]:
lv = vc.load_npz("levels_1800.npz")

def ss_profile(e, v, d):
    ref = lv[f"prior|{v}|{d}"]; x = lv[f"{e}|{v}|{d}"]
    with np.errstate(invalid="ignore", divide="ignore"):
        return 1 - np.sqrt(x[1] / x[0]) / np.sqrt(ref[1] / ref[0]), ref[0]

def draw(lang):
    V = vc.VOCAB[lang]
    fig, axs = plt.subplots(1, 2, figsize=(vc.PAGE_W * 0.8, 0.45 * vc.PAGE_W), sharey=True)
    for ax, d, letter in zip(axs, ["reached", "storm"], "ab"):
        for e in EXPS:
            p, n = ss_profile(e, "ref", d)
            p[n < 100] = np.nan
            ax.plot(p, zkm, color=vc.exp_color(e), label=vc.exp_label(e, lang))
        ax.axvline(0, color="k", lw=0.6)
        ax.set_xlabel(V["ss_rmse"]); ax.set_title(V[f"dom_{d}"])
        vc.panel_label(ax, letter)
    axs[0].set_ylabel(V["height"])
    axs[0].legend(loc="best")
    fig.tight_layout()
    return fig

vc.save(draw, "V1_fig6_profile")

## Resumen numérico

In [ ]:
for e in EXPS:
    r = ss[(ss.exp == e) & (ss.time == "1800") & (ss["var"] == "ref")].set_index("domain")
    line = ", ".join(f"{d} {r.loc[d, 'ss_rmse']:+.2f}" for d in vc.DOMAINS)
    print(f"{vc.exp_label(e, 'es')}: SS_RMSE de Z a las 18 UTC: {line}")
for e in EXPS:
    r = ss[(ss.exp == e) & (ss.domain == "storm") & (ss["var"] == "ref")].sort_values("lead")
    print(f"{vc.exp_label(e, 'es')}: SS_RMSE de Z en la tormenta por plazo: " +
          ", ".join(f"{l} {v:+.2f}" for l, v in zip(r.lead, r.ss_rmse)))